In [4]:
"""
私人厨师智能体模块

基于 LangChain + LangGraph 构建的 AI 智能体，能够根据用户提供的食材照片或清单，
通过联网搜索推荐合适的菜谱，并从营养价值和制作难度两个维度进行评估排序。

核心能力：
- 多模态输入：支持纯文本和图文混合（食材照片+文字描述）
- 联网搜索：通过 Tavily 搜索引擎实时检索菜谱
- 消息持久化：基于 SQLite 的 checkpoint 机制，会话状态跨重启保留
- 消息摘要：当对话轮次过多时自动总结历史，控制上下文长度
- 流式输出：逐 token 返回响应，提升用户交互体验
"""

import asyncio
import os
import sqlite3
from collections.abc import AsyncGenerator
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage
from langchain_tavily import TavilySearch
from langgraph.checkpoint.sqlite import SqliteSaver

# 加载 .env 文件中的环境变量（ZHIPU_BASE_URL、ZHIPU_API_KEY、TAVILY_API_KEY）
load_dotenv()

# 全局 SQLite 连接池和 checkpointer（保证生命周期贯穿应用全程）
_db_path = "checkpoint.db"
_db_connection = sqlite3.connect(
    str(_db_path),
    check_same_thread=False,
    timeout=10,
)
_checkpointer = SqliteSaver(_db_connection)
_checkpointer.setup()

# 智能体系统提示词：定义角色、工作流程和输出规范
system_prompt = """你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份[当前可用食材清单]。
2.智能食谱检索：优先调用 web_search 工具，以[可用食材清单]为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""


def _create_agent():
    """
    创建并配置私人厨师智能体实例。

    组件说明：
    - model: 智谱 GLM-4.1V-Thinking-Flash 模型，支持多模态（图片+文本）输入
    - middleware: 消息摘要中间件，当消息数超过 3 条时自动总结，仅保留最近 1 条原始消息
    - checkpointer: 基于 SQLite 的持久化检查点，用于保存和恢复会话状态
    - tools: Tavily 搜索工具，用于联网检索菜谱信息

    Returns:
        配置完成的 LangChain agent 实例
    """
    # 初始化大语言模型（通过 OpenAI 兼容接口调用智谱 API）
    model = init_chat_model(
        model="GLM-4.1V-Thinking-Flash",
        model_provider="openai",
        base_url=os.getenv("ZHIPU_BASE_URL"),
        api_key=os.getenv("ZHIPU_API_KEY"),
    )

    # 初始化消息摘要中间件：
    # - trigger=("messages", 3): 当会话消息数超过 3 条时触发摘要
    # - keep=("messages", 1): 摘要后仅保留最近 1 条原始消息
    middleware = SummarizationMiddleware(
        model=model,
        trigger=("messages", 3),
        keep=("messages", 1),
    )

    # 初始化 SQLite 持久化检查点（会话状态存储在 checkpoint.db 文件中）
    # 使用全局 checkpointer 确保所有操作共用同一实例
    checkpointer = _checkpointer

    # 初始化 Tavily 搜索工具：每次搜索最多返回 5 条通用主题结果
    tavily_tool = TavilySearch(max_results=5, topic="general")

    # 创建智能体：绑定模型、系统提示词、持久化存储、工具和中间件
    return create_agent(
        model=model,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
        tools=[tavily_tool],
        middleware=[middleware],
    )


# 模块级智能体单例（应用生命周期内复用）
agent = _create_agent()


In [7]:
async def stream_chat(prompt: str, thread_id: str, image: str | None = None) -> AsyncGenerator[str, None]:
    """
    与智能体进行流式对话。

    支持纯文本和图文混合两种输入模式。通过 thread_id 隔离不同会话，
    同一 thread_id 的消息会自动关联历史上下文。

    Args:
        prompt: 用户发送的文本消息
        thread_id: 会话线程 ID，用于标识和隔离不同对话
        image: 可选的图片 URL，支持食材照片等多模态输入

    Yields:
        str: 智能体响应的文本片段（逐 token 流式返回）
    """
    # 构建消息内容：图文混合 or 纯文本
    if image:
        content = [
            {"type": "image_url", "image_url": {"url": image}},
            {"type": "text", "text": prompt},
        ]
    else:
        content = prompt

    multimodal_message = HumanMessage(content=content)

    config = {"configurable": {"thread_id": thread_id}}

    # 使用 asyncio 异步迭代流式输出
    async for token, metadata in agent.astream(
        {"messages": [multimodal_message]},
        stream_mode="messages",
        config=config,
    ):
        if token.content:
            # token.content 可能是字符串或列表（多模态块）
            if isinstance(token.content, str):
                yield token.content
            elif isinstance(token.content, list):
                for block in token.content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        yield block["text"]
                    elif isinstance(block, str):
                        yield block

In [8]:
stream_chat("请问你是什么模型","test11212312")

<async_generator object stream_chat at 0x000001E3F74AA9E0>